# Entrega 2 — Baseline (TF-IDF + LR) vs. BETO fine-tuneado

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MathiasVeraM/proyecto-semestral-ia2/blob/main/notebooks/01_baseline_vs_beto.ipynb)

Clasificación de observaciones de un centro de rescate animal en 0 (Normal), 1 (Riesgo Leve), 2 (Riesgo Alto).

**Protocolo:** mismos splits estratificados 70/15/15 para todos los modelos; hiperparámetros elegidos solo con
*validación*; el *test* se evalúa una vez al final. Métricas: F1 ponderado (principal), F1 macro, recall de la clase 2
y latencia por observación.

> En Colab: Entorno de ejecución → Cambiar tipo de entorno → **GPU (T4)** antes de ejecutar la sección de BETO.

## 0. Configuración del entorno

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MathiasVeraM/proyecto-semestral-ia2.git"

if IN_COLAB:
    if not os.path.exists("proyecto-semestral-ia2"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("proyecto-semestral-ia2")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # ejecución local: subir a la raíz del repo
sys.path.insert(0, os.getcwd())
print("Directorio de trabajo:", os.getcwd())

In [ ]:
import json, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.config import DATA_RAW, SEED, LABELS
from src.baseline import make_majority, make_tfidf_lr, select_C
from src.evaluate import compute_metrics, report_text, plot_confusion, measure_latency
from src.train_beto import BetoConfig, train_beto, predict, make_predict_one, get_device, set_seed

pd.set_option("display.max_colwidth", 110)
COL = {"mayoritaria": "#9aa0a6", "tfidf": "#4c78a8", "beto": "#e07b39"}
Path("results/figures").mkdir(parents=True, exist_ok=True)
Path("results/tables").mkdir(parents=True, exist_ok=True)
print("Dispositivo:", get_device(), "| torch", torch.__version__)

## 1. Datos y preprocesamiento

In [ ]:
# Si no existen los splits, se generan (dataset sintético -> limpieza -> split 70/15/15)
if not DATA_RAW.exists():
    subprocess.run([sys.executable, "-m", "src.generate_dataset"], check=True)
if not Path("data/processed/train.csv").exists():
    subprocess.run([sys.executable, "-m", "src.preprocess"], check=True)

train = pd.read_csv("data/processed/train.csv")
val = pd.read_csv("data/processed/val.csv")
test = pd.read_csv("data/processed/test.csv")
print(json.dumps(json.load(open("data/processed/preprocess_report.json", encoding="utf-8")), indent=2, ensure_ascii=False))

In [ ]:
# Distribución de clases por split
dist = pd.DataFrame({n: d["etiqueta"].value_counts().sort_index() for n, d in
                     [("train", train), ("val", val), ("test", test)]})
dist.index = [LABELS[i] for i in dist.index]
display(dist)
ax = dist.T.plot(kind="bar", figsize=(6, 3.5), color=["#4c78a8", "#f2b134", "#d1495b"], rot=0)
ax.set_ylabel("observaciones"); ax.set_title("Distribución de clases por split")
plt.tight_layout(); plt.savefig("results/figures/distribucion_clases.png", dpi=150); plt.show()

In [ ]:
# Efecto del preprocesamiento (texto original -> limpio)
display(train[["texto_original", "texto", "etiqueta"]].sample(8, random_state=1))

## 2. Baselines

- **Clase mayoritaria:** piso de referencia; muestra lo engañosa que es la *accuracy* con clases desbalanceadas.
- **TF-IDF + Regresión Logística:** bolsa de palabras ponderada (unigramas y bigramas). No modela orden ni negación.
  El hiperparámetro `C` se elige por F1 ponderado en validación. Se entrena solo con *train* (igual que BETO).

In [ ]:
rows = []   # resultados de todos los modelos
preds = {}  # predicciones de test para el análisis de errores

def registrar(nombre, y_val, p_val, y_test, p_test, extra=None):
    for split, y, p in (("val", y_val, p_val), ("test", y_test, p_test)):
        rows.append({"modelo": nombre, "split": split, **compute_metrics(y, p), **(extra or {})})
    preds[nombre] = np.asarray(p_test)

X_tr, y_tr = train["texto"], train["etiqueta"]
X_va, y_va = val["texto"], val["etiqueta"]
X_te, y_te = test["texto"], test["etiqueta"]

# 1) Clase mayoritaria
maj = make_majority().fit(X_tr, y_tr)
registrar("Mayoritaria", y_va, maj.predict(X_va), y_te, maj.predict(X_te))

# 2) TF-IDF + LR (sin pesos de clase) y 3) con pesos balanceados
tfidf_models = {}
for nombre, cw in (("TF-IDF + LR", None), ("TF-IDF + LR (balanceado)", "balanced")):
    C, tabla = select_C(X_tr, y_tr, X_va, y_va, class_weight=cw)
    print(f"{nombre}: F1 val por C = { {k: round(v, 3) for k, v in tabla.items()} } -> C = {C}")
    m = make_tfidf_lr(C, cw).fit(X_tr, y_tr)
    tfidf_models[nombre] = m
    registrar(nombre, y_va, m.predict(X_va), y_te, m.predict(X_te), {"C": C})

In [ ]:
res = pd.DataFrame(rows)
display(res[res.modelo != "BETO"].round(3))
print(report_text(y_te, preds["TF-IDF + LR"]))
fig, ax = plt.subplots(figsize=(4.8, 4))
plot_confusion(y_te, preds["TF-IDF + LR"], "TF-IDF + LR (test)", ax)
plt.tight_layout(); plt.savefig("results/figures/cm_tfidf.png", dpi=150); plt.show()

## 3. BETO fine-tuneado (versión inicial)

`dccuchile/bert-base-spanish-wwm-cased` + cabeza de clasificación lineal (3 clases). Configuración **intencionalmente
simple**: sin pérdida ponderada, sin congelar capas, sin búsqueda de hiperparámetros. Se conserva la época con
mejor F1 en validación. Las mejoras (Weighted Cross-Entropy, búsqueda de lr/épocas, DistilBETO, cuantización/ONNX)
quedan para la entrega final.

In [ ]:
cfg = BetoConfig()
print(json.dumps(cfg.to_dict(), indent=2))
model, tok, hist = train_beto(train, val, cfg)
hist = pd.DataFrame(hist); display(hist.round(4))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].plot(hist.epoca, hist.train_loss, marker="o", label="train", color="#4c78a8")
axs[0].plot(hist.epoca, hist.val_loss, marker="o", label="validación", color="#e07b39")
axs[0].set_xlabel("época"); axs[0].set_ylabel("pérdida"); axs[0].set_title("Curva de pérdida"); axs[0].legend()
axs[1].plot(hist.epoca, hist.val_f1, marker="o", color="#e07b39")
axs[1].set_xlabel("época"); axs[1].set_ylabel("F1 ponderado (val)"); axs[1].set_title("F1 en validación")
plt.tight_layout(); plt.savefig("results/figures/curvas_beto.png", dpi=150); plt.show()
hist.to_csv("results/tables/beto_historial_seed42.csv", index=False)

In [ ]:
p_va, _ = predict(model, tok, X_va, cfg.max_len)
p_te, probs_te = predict(model, tok, X_te, cfg.max_len)
registrar("BETO", y_va, p_va, y_te, p_te)
print(report_text(y_te, p_te))
fig, ax = plt.subplots(figsize=(4.8, 4))
plot_confusion(y_te, p_te, "BETO (test)", ax)
plt.tight_layout(); plt.savefig("results/figures/cm_beto.png", dpi=150); plt.show()

### 3.1 Estabilidad entre semillas

Con pocos datos, el entrenamiento de BETO puede variar bastante según la semilla (inicialización de la cabeza y
orden de los lotes). Se repite con 2 semillas adicionales y se reporta media ± desviación estándar en *test*.

In [ ]:
seed_rows = [{"seed": cfg.seed, **compute_metrics(y_te, p_te)}]
for s in (43, 44):
    c = BetoConfig(seed=s)
    m_s, t_s, _ = train_beto(train, val, c, verbose=False)
    p_s, _ = predict(m_s, t_s, X_te, c.max_len)
    seed_rows.append({"seed": s, **compute_metrics(y_te, p_s)})
    del m_s; torch.cuda.empty_cache()
seeds = pd.DataFrame(seed_rows); display(seeds.round(3))
resumen_seeds = seeds.drop(columns="seed").agg(["mean", "std"]).round(3); display(resumen_seeds)
seeds.to_csv("results/tables/beto_semillas.csv", index=False)

## 4. Comparación y latencia

In [ ]:
lat = {}
lat["TF-IDF + LR"] = measure_latency(lambda t: tfidf_models["TF-IDF + LR"].predict([t])[0], list(X_te))
lat["Mayoritaria"] = measure_latency(lambda t: maj.predict([t])[0], list(X_te))
lat["BETO"] = measure_latency(make_predict_one(model, tok, cfg.max_len), list(X_te))
dev = str(get_device())

res = pd.DataFrame(rows)
test_res = res[res.split == "test"].set_index("modelo")
tabla = test_res[["f1_weighted", "f1_macro", "accuracy", "recall_clase2"]].copy()
tabla["latencia_media_ms"] = pd.Series({k: v["latencia_media_ms"] for k, v in lat.items()})
tabla["latencia_p95_ms"] = pd.Series({k: v["latencia_p95_ms"] for k, v in lat.items()})
tabla = tabla.loc[["Mayoritaria", "TF-IDF + LR", "TF-IDF + LR (balanceado)", "BETO"]].round(3)
print(f"Test | latencia de BETO medida en: {dev}")
display(tabla)
tabla.to_csv("results/tables/comparacion_test.csv")
res.to_csv("results/tables/resultados_val_test.csv", index=False)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
nombres = list(tabla.index)
colores = [COL["mayoritaria"], COL["tfidf"], COL["tfidf"], COL["beto"]]
for ax, met, tit in zip(axs, ["f1_weighted", "recall_clase2"], ["F1 ponderado (test)", "Recall clase 2 / Riesgo Alto (test)"]):
    ax.bar(range(len(nombres)), tabla[met], color=colores)
    ax.set_xticks(range(len(nombres))); ax.set_xticklabels(nombres, rotation=20, ha="right")
    ax.set_ylim(0, 1.05); ax.set_title(tit)
    for i, v in enumerate(tabla[met]): ax.text(i, v + 0.015, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout(); plt.savefig("results/figures/comparacion_modelos.png", dpi=150); plt.show()

## 5. Análisis de errores

Interesa ver dónde BETO y TF-IDF fallan, sobre todo en frases con **negación** ("no presenta sangrado") y en
casos **ambiguos** o **mixtos** ("come bien pero cojea").

In [ ]:
err = test[["texto", "etiqueta"]].copy()
err["pred_tfidf"] = preds["TF-IDF + LR"]; err["pred_beto"] = preds["BETO"]
neg = err.texto.str.contains(r"\b(?:no|ni|sin|nada|ningún|ninguna|descart\w+|ya no)\b", regex=True)
print(f"Subconjunto con negación: {neg.sum()} de {len(err)} observaciones de test")
for nombre, col in (("TF-IDF + LR", "pred_tfidf"), ("BETO", "pred_beto")):
    for etiqueta, sub in (("con negación", err[neg]), ("sin negación", err[~neg])):
        print(f"  {nombre:12s} accuracy {etiqueta}: {(sub[col] == sub.etiqueta).mean():.3f}  (n={len(sub)})")
print("\nErrores de BETO:")
display(err[err.pred_beto != err.etiqueta])
print("Errores de TF-IDF + LR:")
display(err[err.pred_tfidf != err.etiqueta])

## 6. Discusión preliminar (completar con los resultados de la ejecución)

Completar tras ejecutar, con evidencia de las tablas y figuras de arriba:

- **¿BETO supera al baseline?** ¿Por cuánto en F1 ponderado y en recall de la clase 2? ¿Es significativo frente a la
  variación entre semillas (§3.1)?
- **Sobreajuste:** ¿la pérdida de validación sube mientras la de entrenamiento baja (curvas)? ¿En qué época?
- **Desbalance:** ¿qué clase se confunde más (matrices de confusión)? ¿Mejora `class_weight="balanced"` en el baseline?
- **Negación y ambigüedad:** ¿qué modelo falla más en frases con negación o mixtas (§5)?
- **Estabilidad y cómputo:** desviación entre semillas, tiempo de entrenamiento y latencia (§4).
- **Limitaciones:** dataset sintético con frases de plantilla repetidas (posible optimismo) y test pequeño.